# Fase 3 - Majority Voting: LR + RF + SVM
Ensamble por mayoría de votos para cada combinación de:
- Preprocesamiento: normal | stem | lemma  (parámetro `PREP`)
- Representación: TF-IDF | GloVe (GloVe solo para PREP='normal')
- Dataset: combinado | mx | es | cu

Los modelos se cargan desde `data/modelos/{PREP}/` según el preprocesamiento seleccionado.

## 1. Importaciones

In [40]:
import pandas as pd
import numpy as np
import joblib, os, time
import gzip, shutil
from scipy.stats import mode
from sklearn.metrics import (
    f1_score, accuracy_score,
    precision_score, recall_score, classification_report
)

## 2. Configuración
**Cambiar `PREP` para seleccionar la variante:**
`'normal'` | `'stem'` | `'lemma'`

In [41]:
# ══════════════════════════════════════════════════════
PREP = 'lemma'   # opciones: 'normal' | 'stem' | 'lemma'
# ══════════════════════════════════════════════════════

SUFIJO     = '' if PREP == 'normal' else f'_{PREP}'
DATA_DIR   = '../data'
MODELS_DIR = f'../data/modelos/{PREP}'
GLOVE_VEC  = '../glove-sbwc.i25.vec'
GLOVE_GZ   = '../glove-sbwc.i25.vec.gz'

FEATURE_COLS     = ['n_exc','n_int','n_may','n_emo','n_ris',
                    'n_neg','n_elo','n_com','n_pun']
DATASETS_NOMBRES = ['combinado', 'mx', 'es', 'cu']

print(f'PREP seleccionado : {PREP}')
print(f'Carpeta modelos   : {MODELS_DIR}')

PREP seleccionado : lemma
Carpeta modelos   : ../data/modelos/lemma


## 3. Carga de datos de test

In [42]:
TESTS = {
    'combinado': pd.read_csv(f'{DATA_DIR}/test_clean{SUFIJO}.csv'),
    'mx':        pd.read_csv(f'{DATA_DIR}/test_clean{SUFIJO}_mx.csv'),
    'es':        pd.read_csv(f'{DATA_DIR}/test_clean{SUFIJO}_es.csv'),
    'cu':        pd.read_csv(f'{DATA_DIR}/test_clean{SUFIJO}_cu.csv'),
}

print(f'Datasets de test cargados ({PREP}):')
for nombre, df in TESTS.items():
    print(f'  {nombre:12} {len(df):,} registros')

Datasets de test cargados (lemma):
  combinado    1,800 registros
  mx           600 registros
  es           600 registros
  cu           600 registros


## 4. Carga de GloVe
Solo se carga para PREP='normal'. Para stem y lemma GloVe se omite.

In [43]:
glove = None

if PREP != 'normal':
    print(f'GloVe omitido para PREP="{PREP}" (incompatible con stem/lemma).')
else:
    if not os.path.exists(GLOVE_VEC):
        print('Descomprimiendo GloVe...')
        with gzip.open(GLOVE_GZ, 'rb') as f_in:
            with open(GLOVE_VEC, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)

    class GloVeModel:
        def __init__(self, filepath):
            self.key_to_index = {}
            vectors_list = []
            expected_dim = None
            print('Cargando vectores GloVe...')
            t0 = time.time()
            with open(filepath, 'r', encoding='utf-8') as f:
                for i, line in enumerate(f):
                    if i % 100000 == 0 and i > 0:
                        print(f'  {i:,} vectores...', end='\r')
                    parts = line.rstrip().split(' ')
                    if len(parts) < 2:
                        continue
                    if len(parts) == 2:
                        try:
                            int(parts[0]); int(parts[1])
                            continue
                        except ValueError:
                            pass
                    word = parts[0]
                    try:
                        vector = np.array(parts[1:], dtype=np.float32)
                    except ValueError:
                        continue
                    if expected_dim is None:
                        expected_dim = len(vector)
                    if len(vector) != expected_dim:
                        continue
                    self.key_to_index[word] = len(vectors_list)
                    vectors_list.append(vector)
            self.vectors     = np.vstack(vectors_list)
            self.vector_size = self.vectors.shape[1]
            print(f'\nListo en {time.time()-t0:.1f}s | '
                  f'{len(self.key_to_index):,} vectores | '
                  f'{self.vector_size} dimensiones')
        def __contains__(self, w): return w in self.key_to_index
        def __getitem__(self, w):  return self.vectors[self.key_to_index[w]]

    glove = GloVeModel(GLOVE_VEC)

def vectorizar_glove(textos, modelo, dim=300):
    vectores = []
    for texto in textos:
        palabras = str(texto).split()
        vecs = [modelo[w] for w in palabras if w in modelo]
        vectores.append(np.mean(vecs, axis=0) if vecs else np.zeros(dim))
    return np.array(vectores)

GloVe omitido para PREP="lemma" (incompatible con stem/lemma).


## 5. Funciones de voting y métricas

In [44]:
def majority_vote(pred_lr, pred_rf, pred_svm):
    stacked = np.vstack([pred_lr, pred_rf, pred_svm]).T
    votos, _ = mode(stacked, axis=1, keepdims=False)
    return votos.flatten()

def metricas(y_true, y_pred, modelo, ds, repr_n, prep):
    return {
        'prep':       prep,
        'dataset':    ds,
        'repr':       repr_n,
        'modelo':     modelo,
        'accuracy':   round(accuracy_score(y_true, y_pred), 4),
        'f1_macro':   round(f1_score(y_true, y_pred, average='macro'), 4),
        'f1_ironico': round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        'p_ironico':  round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        'r_ironico':  round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
    }

print('Funciones listas.')

Funciones listas.


## 6. Voting TF-IDF

In [45]:
RESULTADOS = []

for ds in DATASETS_NOMBRES:
    df_te = TESTS[ds]
    X_te  = df_te[['MESSAGE_CLEAN'] + FEATURE_COLS]
    y_te  = df_te['IS_IRONIC'].values

    lr  = joblib.load(f'{MODELS_DIR}/ml_tf-idf_lr_{ds}.pkl')
    rf  = joblib.load(f'{MODELS_DIR}/ml_tf-idf_rf_{ds}.pkl')
    svm = joblib.load(f'{MODELS_DIR}/ml_tf-idf_svm_{ds}.pkl')

    pred_lr  = lr.predict(X_te)
    pred_rf  = rf.predict(X_te)
    pred_svm = svm.predict(X_te)
    pred_vot = majority_vote(pred_lr, pred_rf, pred_svm)

    print(f'\n=== TF-IDF | {ds.upper()} | PREP={PREP} ===')
    for nombre, pred in [('LR',pred_lr),('RF',pred_rf),
                          ('SVM',pred_svm),('VOTING',pred_vot)]:
        m = metricas(y_te, pred, nombre, ds, 'TF-IDF', PREP)
        RESULTADOS.append(m)
        print(f'  {nombre:8} F1-Macro={m["f1_macro"]:.4f} '
              f'Accuracy={m["accuracy"]:.4f} '
              f'F1-Irónico={m["f1_ironico"]:.4f}')


=== TF-IDF | COMBINADO | PREP=lemma ===
  LR       F1-Macro=0.6622 Accuracy=0.6956 F1-Irónico=0.5559
  RF       F1-Macro=0.6569 Accuracy=0.6833 F1-Irónico=0.5615
  SVM      F1-Macro=0.6648 Accuracy=0.6983 F1-Irónico=0.5589
  VOTING   F1-Macro=0.6648 Accuracy=0.6983 F1-Irónico=0.5589

=== TF-IDF | MX | PREP=lemma ===
  LR       F1-Macro=0.6742 Accuracy=0.7133 F1-Irónico=0.5612
  RF       F1-Macro=0.6384 Accuracy=0.6750 F1-Irónico=0.5232
  SVM      F1-Macro=0.6748 Accuracy=0.7150 F1-Irónico=0.5604
  VOTING   F1-Macro=0.6719 Accuracy=0.7117 F1-Irónico=0.5575

=== TF-IDF | ES | PREP=lemma ===
  LR       F1-Macro=0.7135 Accuracy=0.7450 F1-Irónico=0.6185
  RF       F1-Macro=0.6883 Accuracy=0.7333 F1-Irónico=0.5699
  SVM      F1-Macro=0.7164 Accuracy=0.7383 F1-Irónico=0.6374
  VOTING   F1-Macro=0.7208 Accuracy=0.7500 F1-Irónico=0.6305

=== TF-IDF | CU | PREP=lemma ===
  LR       F1-Macro=0.6600 Accuracy=0.7100 F1-Irónico=0.5297
  RF       F1-Macro=0.6721 Accuracy=0.7250 F1-Irónico=0.5404
  S

## 7. Voting GloVe (solo PREP='normal')

In [46]:
if PREP != 'normal':
    print(f'Voting GloVe omitido para PREP="{PREP}".')
else:
    for ds in DATASETS_NOMBRES:
        df_te = TESTS[ds]
        y_te  = df_te['IS_IRONIC'].values

        glove_te = vectorizar_glove(df_te['MESSAGE_CLEAN'], glove)
        X_te = np.hstack([glove_te, df_te[FEATURE_COLS].values.astype(float)])

        lr  = joblib.load(f'{MODELS_DIR}/ml_glove_lr_{ds}.pkl')
        rf  = joblib.load(f'{MODELS_DIR}/ml_glove_rf_{ds}.pkl')
        svm = joblib.load(f'{MODELS_DIR}/ml_glove_svm_{ds}.pkl')

        pred_lr  = lr.predict(X_te)
        pred_rf  = rf.predict(X_te)
        pred_svm = svm.predict(X_te)
        pred_vot = majority_vote(pred_lr, pred_rf, pred_svm)

        print(f'\n=== GloVe | {ds.upper()} | PREP=normal ===')
        for nombre, pred in [('LR',pred_lr),('RF',pred_rf),
                              ('SVM',pred_svm),('VOTING',pred_vot)]:
            m = metricas(y_te, pred, nombre, ds, 'GloVe', PREP)
            RESULTADOS.append(m)
            print(f'  {nombre:8} F1-Macro={m["f1_macro"]:.4f} '
                  f'Accuracy={m["accuracy"]:.4f} '
                  f'F1-Irónico={m["f1_ironico"]:.4f}')

Voting GloVe omitido para PREP="lemma".


## 8. Tabla comparativa y exportación

In [47]:
df_vot = pd.DataFrame(RESULTADOS)

print(f'\nTABLA COMPLETA — PREP={PREP}')
print(df_vot.to_string(index=False))

print(f'\n\nRESUMEN: ¿Voting mejora al mejor modelo individual? (PREP={PREP})')
print('-' * 70)
for repr_n in ['TF-IDF'] + (['GloVe'] if PREP == 'normal' else []):
    for ds in DATASETS_NOMBRES:
        sub = df_vot[(df_vot['dataset']==ds) & (df_vot['repr']==repr_n)]
        mejor_ind = sub[sub['modelo'] != 'VOTING']['f1_macro'].max()
        voting    = sub[sub['modelo'] == 'VOTING']['f1_macro'].values[0]
        delta     = voting - mejor_ind
        signo     = '▲' if delta > 0 else ('▼' if delta < 0 else '=')
        print(f'  {repr_n:8} {ds:12} '
              f'mejor_ind={mejor_ind:.4f} '
              f'voting={voting:.4f} '
              f'{signo} {delta:+.4f}')

out_csv = f'{DATA_DIR}/resultados_voting_{PREP}.csv'
df_vot.to_csv(out_csv, index=False)
print(f'\nGuardado: {out_csv}')


TABLA COMPLETA — PREP=lemma
 prep   dataset   repr modelo  accuracy  f1_macro  f1_ironico  p_ironico  r_ironico
lemma combinado TF-IDF     LR    0.6956    0.6622      0.5559     0.5402     0.5726
lemma combinado TF-IDF     RF    0.6833    0.6569      0.5615     0.5207     0.6093
lemma combinado TF-IDF    SVM    0.6983    0.6648      0.5589     0.5443     0.5743
lemma combinado TF-IDF VOTING    0.6983    0.6648      0.5589     0.5443     0.5743
lemma        mx TF-IDF     LR    0.7133    0.6742      0.5612     0.5699     0.5528
lemma        mx TF-IDF     RF    0.6750    0.6384      0.5232     0.5095     0.5377
lemma        mx TF-IDF    SVM    0.7150    0.6748      0.5604     0.5737     0.5477
lemma        mx TF-IDF VOTING    0.7117    0.6719      0.5575     0.5677     0.5477
lemma        es TF-IDF     LR    0.7450    0.7135      0.6185     0.6169     0.6200
lemma        es TF-IDF     RF    0.7333    0.6883      0.5699     0.6163     0.5300
lemma        es TF-IDF    SVM    0.7383    0.71

## 9. Comparativa global entre preprocesamiento (ejecutar tras correr los 3 PREP)

In [48]:
# Carga los tres CSVs de voting si existen y muestra comparativa
dfs = []
for p in ['normal', 'stem', 'lemma']:
    fpath = f'{DATA_DIR}/resultados_voting_{p}.csv'
    if os.path.exists(fpath):
        dfs.append(pd.read_csv(fpath))
    else:
        print(f'  Falta: {fpath} (ejecuta PREP="{p}" primero)')

if dfs:
    df_global = pd.concat(dfs, ignore_index=True)

    # Mejor resultado por prep × dataset × repr para VOTING e individual
    print('\nMEJOR F1-MACRO POR PREP × DATASET (TF-IDF, VOTING vs mejor individual):\n')
    print(f'{"Prep":8} {"Dataset":12} {"Mejor ind":12} {"Modelo":6} {"Voting":10} {"Δ":8}')
    print('-' * 62)

    for prep in ['normal', 'stem', 'lemma']:
        for ds in DATASETS_NOMBRES:
            sub = df_global[
                (df_global['prep']==prep) &
                (df_global['dataset']==ds) &
                (df_global['repr']=='TF-IDF')
            ]
            if sub.empty:
                continue
            ind = sub[sub['modelo'] != 'VOTING']
            idx_mejor = ind['f1_macro'].idxmax()
            mejor_f1    = ind.loc[idx_mejor, 'f1_macro']
            mejor_mod   = ind.loc[idx_mejor, 'modelo']
            voting_f1   = sub[sub['modelo']=='VOTING']['f1_macro'].values[0]
            delta = voting_f1 - mejor_f1
            signo = '▲' if delta > 0 else ('▼' if delta < 0 else '=')
            print(f'{prep:8} {ds:12} {mejor_f1:>12.4f} {mejor_mod:>6} '
                  f'{voting_f1:>10.4f} {signo}{delta:>+7.4f}')
        print()
else:
    print('Ejecuta el notebook con los 3 valores de PREP para ver la comparativa global.')


MEJOR F1-MACRO POR PREP × DATASET (TF-IDF, VOTING vs mejor individual):

Prep     Dataset      Mejor ind    Modelo Voting     Δ       
--------------------------------------------------------------
normal   combinado          0.6619     LR     0.6611 ▼-0.0008
normal   mx                 0.6924     LR     0.6932 ▲+0.0008
normal   es                 0.7227    SVM     0.7180 ▼-0.0047
normal   cu                 0.6648     LR     0.6641 ▼-0.0007

stem     combinado          0.6737     LR     0.6732 ▼-0.0005
stem     mx                 0.6727    SVM     0.6686 ▼-0.0041
stem     es                 0.7318     RF     0.7133 ▼-0.0185
stem     cu                 0.6796     RF     0.6732 ▼-0.0064

lemma    combinado          0.6648    SVM     0.6648 =+0.0000
lemma    mx                 0.6748    SVM     0.6719 ▼-0.0029
lemma    es                 0.7164    SVM     0.7208 ▲+0.0044
lemma    cu                 0.6721     RF     0.6526 ▼-0.0195

